# MMLandmarks — Exploratory Data Analysis

Multimodal landmark dataset with ground images, satellite images, and text descriptions.  
Splits: **train**, **query**, **index** — each with manifest CSVs and hex-sharded image/text folders.  
Downstream task: geo-localization (GeoClip baseline) — lat/lon quality and category balance matter most.

## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import json
import random

plt.rcParams.update({"figure.dpi": 120})
random.seed(42)
np.random.seed(42)

DATA_ROOT = Path("../../data/MML_Data")
assert DATA_ROOT.exists(), f"DATA_ROOT not found: {DATA_ROOT.resolve()}"
print(f"DATA_ROOT resolved to: {DATA_ROOT.resolve()}")

In [ ]:
# 3-level hex-prefix sharding (confirmed from filesystem)
# ground: .jpg | satellite: .png | text: .json
EXTENSIONS = {"ground": "jpg", "satellite": "png", "text": "json"}


def asset_path(split, modality, hex_id):
    ext = EXTENSIONS[modality]
    return DATA_ROOT / split / modality / hex_id[0] / hex_id[1] / hex_id[2] / f"{hex_id}.{ext}"

## 2. Split Sizes

In [ ]:
split_files = {
    ("train", "master"): DATA_ROOT / "train" / "mml_train.csv",
    ("train", "ground"): DATA_ROOT / "train" / "mml_train_ground.csv",
    ("train", "satellite"): DATA_ROOT / "train" / "mml_train_satellite.csv",
    ("train", "text"): DATA_ROOT / "train" / "mml_train_text.csv",
    ("index", "ground"): DATA_ROOT / "index" / "mml_index_ground.csv",
    ("index", "satellite"): DATA_ROOT / "index" / "mml_index_satellite.csv",
    ("query", "master"): DATA_ROOT / "query" / "mml_query.csv",
    ("query", "ground"): DATA_ROOT / "query" / "mml_query_ground.csv",
    ("query", "satellite"): DATA_ROOT / "query" / "mml_query_satellite.csv",
    ("query", "text"): DATA_ROOT / "query" / "mml_query_text.csv",
}

row_counts = []
for (split, modality), path in split_files.items():
    if path.exists():
        n = sum(1 for _ in open(path)) - 1  # subtract header
        row_counts.append({"split": split, "modality": modality, "rows": n, "file": path.name})
    else:
        row_counts.append({"split": split, "modality": modality, "rows": "MISSING", "file": path.name})

df_counts = pd.DataFrame(row_counts)
display(df_counts.pivot_table(index="split", columns="modality", values="rows", aggfunc="first"))

## 3. Manifest Schemas

In [ ]:
manifests = {}
for (split, modality), path in split_files.items():
    if path.exists():
        df = pd.read_csv(path)
        key = f"{split}_{modality}"
        manifests[key] = df
        print(f"\n{'='*60}")
        print(f"{key}  ({len(df)} rows)")
        print(f"Columns: {list(df.columns)}")
        print(f"Dtypes:\n{df.dtypes}")
        display(df.head(3))

In [ ]:
# Join key check — only manifests with landmark_id
for split in ["train", "query"]:
    keys_in_split = []
    for k, df in manifests.items():
        if k.startswith(split) and "landmark_id" in df.columns:
            keys_in_split.append((k, set(df["landmark_id"])))
    if len(keys_in_split) >= 2:
        base_name, base_ids = keys_in_split[0]
        for other_name, other_ids in keys_in_split[1:]:
            overlap = len(base_ids & other_ids)
            print(f"{base_name} \u2229 {other_name}: {overlap} / {len(base_ids)} ({overlap/len(base_ids)*100:.1f}%)")

# Note: index manifests have NO landmark_id
print("\nIndex manifests (no landmark_id):")
for k in ["index_ground", "index_satellite"]:
    if k in manifests:
        print(f"  {k} columns: {list(manifests[k].columns)}")

## 4. Metadata Analysis

In [ ]:
meta = pd.read_csv(DATA_ROOT / "additional_info" / "mmlandmarks.csv")
print(f"Metadata shape: {meta.shape}")
print(f"Columns: {list(meta.columns)}")
display(meta.head())

In [ ]:
print("Null counts:")
nulls = meta.isnull().sum()
display(nulls[nulls > 0].to_frame("nulls"))

geo_cols = ["lat", "lon", "min_lat", "min_lon", "max_lat", "max_lon"]
print("\nGeo column statistics:")
display(meta[geo_cols].describe())

## 5. Category Distribution

In [ ]:
cat_counts = meta["category"].value_counts()

fig, ax = plt.subplots(figsize=(12, 6))
cat_counts.head(30).plot.bar(ax=ax, color="steelblue")
ax.set_yscale("log")
ax.set_ylabel("Landmark count (log)")
ax.set_title("Top-30 Categories")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

# Imbalance metrics
top5_share = cat_counts.head(5).sum() / cat_counts.sum()
print(f"Top-5 category share: {top5_share:.1%}")
print(f"Total categories: {len(cat_counts)}")

# Gini coefficient
vals = np.sort(cat_counts.values).astype(float)
n = len(vals)
idx = np.arange(1, n + 1)
gini = (2 * np.sum(idx * vals) / (n * np.sum(vals))) - (n + 1) / n
print(f"Gini coefficient: {gini:.3f}")

## 6. Geographic Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Density hexbin
ax = axes[0]
hb = ax.hexbin(meta["lon"], meta["lat"], gridsize=60, cmap="YlOrRd", mincnt=1)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Landmark Density")
fig.colorbar(hb, ax=ax, label="Count")

# Color by top-8 categories
ax = axes[1]
top_cats = cat_counts.head(8).index.tolist()
for cat in top_cats:
    subset = meta[meta["category"] == cat]
    ax.scatter(subset["lon"], subset["lat"], s=3, alpha=0.4, label=cat)
other = meta[~meta["category"].isin(top_cats)]
ax.scatter(other["lon"], other["lat"], s=1, alpha=0.1, color="grey", label="other")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Top-8 Categories")
ax.legend(fontsize=7, markerscale=3, loc="lower left")

fig.tight_layout()
plt.show()

# Outlier check
outliers = meta[(meta["lat"].abs() > 85) | (meta["lon"].abs() > 180)]
print(f"Outliers (|lat|>85 or |lon|>180): {len(outliers)}")
if len(outliers) > 0:
    display(outliers[["landmark_id", "CommonsCategory", "lat", "lon"]].head(10))

## 7. Images per Landmark

In [ ]:
def count_images(images_str):
    if pd.isna(images_str):
        return 0
    return len(str(images_str).split())


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, modality in enumerate(["ground", "satellite"]):
    key = f"train_{modality}"
    if key not in manifests:
        continue
    df = manifests[key].copy()
    df["n_images"] = df["images"].apply(count_images)

    print(f"{key}: median={df['n_images'].median():.0f}, "
          f"mean={df['n_images'].mean():.1f}, "
          f"min={df['n_images'].min()}, max={df['n_images'].max()}")

    ax = axes[i]
    ax.hist(df["n_images"], bins=50, color="steelblue", edgecolor="white")
    ax.set_yscale("log")
    ax.set_xlabel("Images per landmark")
    ax.set_ylabel("Count (log)")
    ax.set_title(f"Train \u2014 {modality}")

fig.tight_layout()
plt.show()

## 8. Multimodal Completeness (Train Split)

In [ ]:
train_ids = set(manifests["train_master"]["landmark_id"])
has_ground = set(manifests["train_ground"]["landmark_id"])
has_satellite = set(manifests["train_satellite"]["landmark_id"])
has_text = set(manifests["train_text"]["landmark_id"])
has_all = has_ground & has_satellite & has_text

total = len(train_ids)
print(f"Train landmarks total:       {total}")
print(f"  with ground images:        {len(has_ground)} ({len(has_ground)/total:.1%})")
print(f"  with satellite images:     {len(has_satellite)} ({len(has_satellite)/total:.1%})")
print(f"  with text:                 {len(has_text)} ({len(has_text)/total:.1%})")
print(f"  with ALL three modalities: {len(has_all)} ({len(has_all)/total:.1%})")

# Landmarks missing a modality
missing_ground = train_ids - has_ground
missing_sat = train_ids - has_satellite
missing_text = train_ids - has_text
print(f"\n  missing ground: {len(missing_ground)}, missing satellite: {len(missing_sat)}, missing text: {len(missing_text)}")

## 9. Bounding Box Precision

In [ ]:
meta["bbox_area"] = (meta["max_lat"] - meta["min_lat"]) * (meta["max_lon"] - meta["min_lon"])

fig, ax = plt.subplots(figsize=(10, 5))
reasonable = meta["bbox_area"].dropna()
ax.hist(reasonable, bins=100, color="steelblue", edgecolor="white")
ax.set_yscale("log")
ax.set_xlabel("Bounding box area (deg\u00b2)")
ax.set_ylabel("Count (log)")
ax.set_title("Bounding Box Area Distribution")
fig.tight_layout()
plt.show()

print(f"Bbox area \u2014 median: {reasonable.median():.6f}, mean: {reasonable.mean():.6f}, max: {reasonable.max():.4f}")

# Flag suspiciously large bboxes (> 99th percentile)
threshold = reasonable.quantile(0.99)
large_bbox = meta[meta["bbox_area"] > threshold]
print(f"\nLandmarks with bbox area > {threshold:.4f} (99th pctl): {len(large_bbox)}")
display(large_bbox[["landmark_id", "CommonsCategory", "lat", "lon", "bbox_area"]].head(10))

## 10. Sample Images

In [ ]:
def get_random_image_ids(manifest_df, n=9):
    all_ids = []
    for images_str in manifest_df["images"].dropna():
        all_ids.extend(str(images_str).split())
    return random.sample(all_ids, min(n, len(all_ids)))


def show_image_grid(split, modality, image_ids, title):
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    for ax, img_id in zip(axes.flat, image_ids):
        p = asset_path(split, modality, img_id)
        if p.exists():
            img = Image.open(p)
            ax.imshow(img)
            ax.set_title(img_id[:12] + "\u2026", fontsize=8)
        else:
            ax.text(0.5, 0.5, f"NOT FOUND\n{p.relative_to(DATA_ROOT)}",
                    ha="center", va="center", fontsize=8)
        ax.axis("off")
    fig.suptitle(title, fontsize=14)
    fig.tight_layout()
    plt.show()

In [ ]:
ground_ids = get_random_image_ids(manifests["train_ground"], 9)
show_image_grid("train", "ground", ground_ids, "Sample Ground Images (Train)")

In [ ]:
sat_ids = get_random_image_ids(manifests["train_satellite"], 9)
show_image_grid("train", "satellite", sat_ids, "Sample Satellite Images (Train)")

## 11. Sample Text

In [ ]:
text_manifest = manifests["train_text"]
sample_text_ids = random.sample(list(text_manifest["json"].dropna().astype(str)), 5)

for tid in sample_text_ids:
    p = asset_path("train", "text", tid.strip())
    print(f"\n{'='*60}")
    print(f"File: {p.name}  (exists={p.exists()})")
    if p.exists():
        content = json.loads(p.read_text(encoding="utf-8"))
        if isinstance(content, dict):
            for k, v in list(content.items())[:5]:
                print(f"  {k}: {str(v)[:200]}")
        else:
            print(f"  {str(content)[:500]}")
    else:
        print("  FILE NOT FOUND")

## 12. Summary

| Metric | Value |
|--------|-------|
| Train landmarks | 17,557 |
| Query landmarks | 1,000 |
| Index ground rows | ~714K |
| Index satellite rows | ~99.5K |
| Metadata landmarks | _fill after running_ |
| Total categories | _fill after running_ |
| Top-5 category share | _fill after running_ |
| Gini coefficient | _fill after running_ |
| Train multimodal complete (%) | _fill after running_ |
| Median ground images/landmark | _fill after running_ |
| Median satellite images/landmark | _fill after running_ |
| Median bbox area (deg\u00b2) | _fill after running_ |

**Key observations (fill after running):**
- Geographic bias: _mostly US/Europe?_
- Category imbalance: _how severe?_
- Multimodal coverage: _any missing modalities?_
- Bbox outliers: _any problematic landmarks?_